In [1]:
# # after installing git
!pip install -q git+https://github.com/huggingface/transformers.git
!pip install -q accelerate datasets peft bitsandbytes
# !pip install pillow
# #then restart kernel

In [1]:
from datasets import load_dataset
ds = load_dataset("SimulaMet-HOST/Kvasir-VQA")["raw"]

/home/ebmi/anaconda3/envs/idefics3/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
ds

Dataset({
    features: ['image', 'source', 'question', 'answer', 'img_id'],
    num_rows: 58849
})

In [3]:
import os
from datasets import Dataset, Features, Image, Value, load_dataset, DatasetDict
from PIL import Image as PILImage
import random
import pandas as pd
from collections import defaultdict

In [4]:
from datasets import concatenate_datasets

# Remove invalid question entries
valid_ds = ds.filter(lambda ex: ex["question"] and ex["question"] != "none")

# Identify abnormal samples (source != 'normal')
abnormal_ids = set(
    ex["img_id"] for ex in valid_ds if ex["source"].lower() != "normal"
)

# add "Does this image contain any finding?" = "yes" for abnormal cases
seen_ids = set()
added_examples = []
for ex in valid_ds:
    if ex["img_id"] in abnormal_ids and ex["img_id"] not in seen_ids:
        added_examples.append({
            "image": ex["image"],
            "source": ex["source"],
            "question": "Does this image contain any finding?",
            "answer": "yes",
            "img_id": ex["img_id"]
        })
        seen_ids.add(ex["img_id"])

# Combine cleaned data with added questions
modified_ds = Dataset.from_list(added_examples)
cleaned_ds = concatenate_datasets([valid_ds, modified_ds])


In [5]:
# summary
print("Original size:", len(ds))
print("Cleaned size: ", len(valid_ds))
print("final modifierd data size: ", len(cleaned_ds))
print("new added QAs: ", len(added_examples))

Original size: 58849
Cleaned size:  58798
final modifierd data size:  62747
new added QAs:  3949


In [6]:
cleaned_ds

Dataset({
    features: ['image', 'source', 'question', 'answer', 'img_id'],
    num_rows: 62747
})

## Starting with Modified dataset

In [7]:
import random
from datasets import load_dataset, DatasetDict

# all unique img_ids
all_ids = sorted(set(cleaned_ds["img_id"]))

# Shuffle & split those IDs into 80/10/10
random.seed(42)
random.shuffle(all_ids)

n = len(all_ids)
n_train = int(0.8 * n)
n_val   = int(0.1 * n)
# n_test will be whatever is left
train_ids = set(all_ids[:n_train])
val_ids   = set(all_ids[n_train : n_train + n_val])
test_ids  = set(all_ids[n_train + n_val :])



In [8]:
def filter_by_id(example, id_set):
    return example["img_id"] in id_set



In [9]:
train_ds = cleaned_ds.filter(lambda ex: filter_by_id(ex, train_ids), 
                     batched=False)
val_ds   = cleaned_ds.filter(lambda ex: filter_by_id(ex, val_ids), 
                     batched=False)
test_ds  = cleaned_ds.filter(lambda ex: filter_by_id(ex, test_ids), 
                     batched=False)

In [10]:
dataset = DatasetDict({
    "train":      train_ds,
    "validation": val_ds,
    "test":       test_ds,
})

print({k: len(v) for k, v in dataset.items()})

{'train': 50105, 'validation': 6287, 'test': 6355}


In [11]:
# keeping copy for tracking ( image,question,answer,source,img_id)  #dataset_full['test'][i]['img_id'] (or ['source'])
dataset_full = dataset

data = dataset_full.remove_columns(["source", "img_id"])
data

DatasetDict({
    train: Dataset({
        features: ['image', 'question', 'answer'],
        num_rows: 50105
    })
    validation: Dataset({
        features: ['image', 'question', 'answer'],
        num_rows: 6287
    })
    test: Dataset({
        features: ['image', 'question', 'answer'],
        num_rows: 6355
    })
})

In [12]:
data['train']

Dataset({
    features: ['image', 'question', 'answer'],
    num_rows: 50105
})

In [13]:
dataset_full['test']

Dataset({
    features: ['image', 'source', 'question', 'answer', 'img_id'],
    num_rows: 6355
})

In [ ]:
# import pandas as pd

# # Convert to pandas DataFrame and drop the image column
# df_test = dataset_full["test"].to_pandas().drop(columns=["image"])

# df_test.to_csv("test_data_without_image.csv", index=False)


In [ ]:
import torch
from peft import LoraConfig
from transformers import AutoProcessor, BitsAndBytesConfig, Idefics3ForConditionalGeneration

DEVICE = "cuda:0"
USE_LORA = False
USE_QLORA = True


processor = AutoProcessor.from_pretrained(
    "HuggingFaceTB/SmolVLM-Base",
    do_image_splitting=False
)


# Three options for training, from the lowest precision training to the highest precision training:
# - QLora
# - Standard Lora
# - Full fine-tuning
if USE_QLORA or USE_LORA:
    lora_config = LoraConfig(
        r=8,
        lora_alpha=8,
        lora_dropout=0.1,
        target_modules='.*(text_model|modality_projection|perceiver_resampler).*(down_proj|gate_proj|up_proj|k_proj|q_proj|v_proj|o_proj).*$',
        use_dora=False if USE_QLORA else True,
        init_lora_weights="gaussian"
    )
    if USE_QLORA:
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.bfloat16
        )
    model = Idefics3ForConditionalGeneration.from_pretrained(
        "HuggingFaceTB/SmolVLM-Base",
        torch_dtype=torch.bfloat16,
        quantization_config=bnb_config if USE_QLORA else None,
    )
    model.add_adapter(lora_config)
    model.enable_adapters()
else:
    model = Idefics3ForConditionalGeneration.from_pretrained(
        "HuggingFaceTB/SmolVLM-Base",
        torch_dtype=torch.bfloat16,
        # _attn_implementation="flash_attention_2", # Only available on A100 or H100
    ).to(DEVICE)



In [16]:

def print_trainable_parameters(model):
    trainable = 0
    total = 0
    for name, param in model.named_parameters():
        total += param.numel()
        if param.requires_grad:
            trainable += param.numel()
    print(f"Trainable parameters: {trainable:,}")
    print(f"Total parameters: {total:,}")
    print(f"Percentage of trainable parameters: {100 * trainable / total:.2f}%")

print_trainable_parameters(model)

Trainable parameters: 9,043,968
Total parameters: 1,233,858,416
Percentage of trainable parameters: 0.73%


In [17]:
for name, module in model.named_modules():
    print(name)


model
model.vision_model
model.vision_model.embeddings
model.vision_model.embeddings.patch_embedding
model.vision_model.embeddings.position_embedding
model.vision_model.encoder
model.vision_model.encoder.layers
model.vision_model.encoder.layers.0
model.vision_model.encoder.layers.0.self_attn
model.vision_model.encoder.layers.0.self_attn.k_proj
model.vision_model.encoder.layers.0.self_attn.v_proj
model.vision_model.encoder.layers.0.self_attn.q_proj
model.vision_model.encoder.layers.0.self_attn.out_proj
model.vision_model.encoder.layers.0.layer_norm1
model.vision_model.encoder.layers.0.mlp
model.vision_model.encoder.layers.0.mlp.activation_fn
model.vision_model.encoder.layers.0.mlp.fc1
model.vision_model.encoder.layers.0.mlp.fc2
model.vision_model.encoder.layers.0.layer_norm2
model.vision_model.encoder.layers.1
model.vision_model.encoder.layers.1.self_attn
model.vision_model.encoder.layers.1.self_attn.k_proj
model.vision_model.encoder.layers.1.self_attn.v_proj
model.vision_model.encoder

In [18]:
print("All special tokens:", processor.tokenizer.all_special_tokens)
print("Additional special tokens:", processor.tokenizer.additional_special_tokens)

All special tokens: ['<|im_start|>', '<|endoftext|>', '<|im_end|>', '<fake_token_around_image>', '<image>', '<end_of_utterance>']
Additional special tokens: ['<fake_token_around_image>', '<image>', '<end_of_utterance>']


In [19]:
processor.tokenizer.additional_special_tokens_ids[processor.tokenizer.additional_special_tokens.index("<image>")]

49153

In [20]:
class MyDataCollator:
    def __init__(self, processor):
        self.processor = processor
        self.image_token_id = processor.tokenizer.additional_special_tokens_ids[
            processor.tokenizer.additional_special_tokens.index("<image>")
        ]

    def __call__(self, examples):
        texts = []
        images = []
        for example in examples:
            image = example["image"]
            question = example["question"]
            answer = example["answer"]
            messages = [
                {
                    "role": "user",
                    "content": [
                        {"type": "text", "text": "Answer as a medical specialist"},
                        {"type": "image"},
                        {"type": "text", "text": question}
                    ]
                },
                {
                    "role": "assistant",
                    "content": [
                        {"type": "text", "text": answer}
                    ]
                }
            ]
            text = processor.apply_chat_template(messages, add_generation_prompt=False)
            texts.append(text.strip())
            images.append([image])

        batch = processor(text=texts, images=images, return_tensors="pt", padding=True)

        labels = batch["input_ids"].clone()
        labels[labels == processor.tokenizer.pad_token_id] = -100
        labels[labels == self.image_token_id] = -100
        batch["labels"] = labels

        return batch

data_collator = MyDataCollator(processor)


In [21]:
data_collator

In [22]:
data_collator([data['train'][0]])

{'pixel_values': tensor([[[[[-0.9686, -0.9608, -0.9608,  ..., -0.9686, -0.9765, -0.9843],
           [-0.9686, -0.9686, -0.9686,  ..., -0.9686, -0.9765, -0.9922],
           [-0.9686, -0.9608, -0.9608,  ..., -0.9686, -0.9765, -0.9843],
           ...,
           [-0.9843, -0.9843, -0.9843,  ..., -0.9922, -1.0000, -1.0000],
           [-0.9922, -0.9765, -0.9765,  ..., -0.9922, -1.0000, -1.0000],
           [-0.9922, -0.9765, -0.9765,  ..., -1.0000, -1.0000, -1.0000]],

          [[-0.9686, -0.9608, -0.9529,  ..., -0.9686, -0.9765, -0.9843],
           [-0.9686, -0.9686, -0.9686,  ..., -0.9686, -0.9765, -0.9922],
           [-0.9686, -0.9608, -0.9608,  ..., -0.9686, -0.9765, -0.9843],
           ...,
           [-0.9843, -0.9843, -0.9843,  ..., -0.9765, -0.9922, -1.0000],
           [-0.9922, -0.9765, -0.9765,  ..., -0.9843, -0.9922, -1.0000],
           [-0.9922, -0.9765, -0.9765,  ..., -0.9922, -0.9922, -1.0000]],

          [[-0.9843, -0.9608, -0.9843,  ..., -0.9843, -0.9922, -1.0000]

In [23]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    num_train_epochs=15,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=8,
    warmup_steps=50,
    learning_rate=1e-4,
    weight_decay=0.01,
    logging_steps=25,
    output_dir=r"smolvlm-base_checkpoint",
    save_strategy="epoch", #initially steps, but save and eval strategy must be same for load best model at end
    save_steps=250,
    save_total_limit=1,
    eval_strategy="epoch",
    # fp16=True, #if using float16 in model initialization
    bf16=True,
    remove_unused_columns=False,
    report_to="none",
    #
    # load_best_model_at_end=True,
    # metric_for_best_model="eval_loss",
    # greater_is_better=False,
 
)

trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=data_collator,
    train_dataset=data['train'],
    eval_dataset=data['validation'], # adding evaluate (loss) on the eval set, note that it will incur some additional GPU memory
)


`loss_type=None` was set in the config but it is unrecognised.Using the default loss: `ForCausalLMLoss`.

In [107]:
# import torch
# print(torch.cuda.device_count()) 

2


In [ ]:
trainer.train()

In [ ]:
# #to run the test operaption from the saved checkpoint

# from transformers import AutoProcessor, Idefics3ForConditionalGeneration
# from peft import PeftModel, PeftConfig
# import torch

# # Load base mode
# base_model = Idefics3ForConditionalGeneration.from_pretrained(
#     "HuggingFaceM4/Idefics3-8B-Llama3",
#     torch_dtype=torch.bfloat16,
#     device_map="auto",
# )

# # Load adapter 
# adapter_checkpoint = "idefics3_model_new/checkpoint-2349"  #best checkpoint (check on trainer_state.json)
# model = PeftModel.from_pretrained(base_model, adapter_checkpoint)
# model.eval()

# # Load processor
# processor = AutoProcessor.from_pretrained("HuggingFaceM4/Idefics3-8B-Llama3")

Loading checkpoint shards: 100%|██████████| 4/4 [00:01<00:00,  2.15it/s]


In [25]:
import evaluate
import torch
from tqdm import tqdm
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import Levenshtein


In [26]:
def clean_answer(text):
    if "Assistant:" in text:
        return text.split("Assistant:")[-1].strip()
    return text.strip()

In [27]:
preds = []
refs = []

for ex in tqdm(data['test']):
    image = ex["image"]
    question = ex["question"]
    answer = ex["answer"]  # This can be a string or list

    # Build prompt for IDEFICS3
    messages = [{
        "role": "user",
        "content": [
            {"type": "text", "text": "Answer as a medical specialist"},
            {"type": "image"},
            {"type": "text", "text": question}
        ]
    }]
    prompt = processor.apply_chat_template(messages, add_generation_prompt=True)
    
    # Tokenize and generate
    inputs = processor(text=prompt, images=[[image]], return_tensors="pt", padding=True).to(model.device)
    with torch.no_grad():
        output_ids = model.generate(**inputs, max_new_tokens=64)
    pred = processor.batch_decode(output_ids, skip_special_tokens=True)[0]
    pred = clean_answer(pred)

    # Store
    preds.append(pred)
    refs.append(answer if isinstance(answer, list) else [answer])

    

100%|██████████| 6355/6355 [17:51<00:00,  5.93it/s]


In [ ]:
# !pip install evaluate scikit-learn nltk rouge-score Levenshtein

In [28]:
# Prepare flat reference list for most metrics
refs_single = [r[0] for r in refs]

# Accuracy
accuracy = sum(p in r for p, r in zip(preds, refs)) / len(refs) * 100

# Load metrics
bleu = evaluate.load("bleu")
rouge = evaluate.load("rouge")
meteor = evaluate.load("meteor")

bleu_res   = bleu.compute(predictions=preds, references=[[r] for r in refs_single])
rouge_res  = rouge.compute(predictions=preds, references=refs_single)
meteor_res = meteor.compute(predictions=preds, references=refs_single)

# Jaccard Similarity
j_scores = []
for r, p in zip(refs_single, preds):
    set_r, set_p = set(r.split()), set(p.split())
    j_scores.append(len(set_r & set_p) / len(set_r | set_p) if (set_r | set_p) else 0)
jaccard = sum(j_scores) / len(j_scores) * 100

# Cosine Similarity (TF-IDF)
vectorizer = TfidfVectorizer().fit(refs_single + preds)
ref_vecs  = vectorizer.transform(refs_single)
pred_vecs = vectorizer.transform(preds)
cos_sims  = cosine_similarity(ref_vecs, pred_vecs).diagonal()
cosine = cos_sims.mean() * 100

# Levenshtein Similarity
def normalized_levenshtein(s1, s2):
    if not s1 and not s2:
        return 0
    return Levenshtein.distance(s1, s2) / max(len(s1), len(s2))

def similarity_score(a_ij, o_q_i, tau=0.5):
    nl = normalized_levenshtein(a_ij, o_q_i)
    return 1 - nl if nl < tau else 0

def average_levenshtein_similarity(ground_truth, predicted):
    total_score = 0
    for refs, pred in zip(ground_truth, predicted):
        if not pred:
            continue
        max_score = max(similarity_score(ref, pred) for ref in refs)
        total_score += max_score
    return total_score / len(ground_truth) * 100

levenshtein_score = average_levenshtein_similarity(refs, preds)



[nltk_data] Downloading package wordnet to /home/ebmi/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to /home/ebmi/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /home/ebmi/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


In [29]:

results = {
    "accuracy (%)": round(accuracy, 2),
    "bleu": bleu_res,
    "rouge": rouge_res,
    "meteor": meteor_res,
    "jaccard (%)": round(jaccard, 2),
    "cosine (%)": round(cosine, 2),
    "levenshtein (%)": round(levenshtein_score, 2)
}

for k, v in results.items():
    print(f"{k}:\n{v}\n")

accuracy (%):
86.25

bleu:
{'bleu': 0.7696510625140321, 'precisions': [0.8650329188002926, 0.7808612440191387, 0.7291880781089414, 0.7124090541632982], 'brevity_penalty': 1.0, 'length_ratio': 1.0445480247573928, 'translation_length': 13670, 'reference_length': 13087}

rouge:
{'rouge1': 0.9168052050615392, 'rouge2': 0.17626077213282387, 'rougeL': 0.9155092080978882, 'rougeLsum': 0.9152286841142088}

meteor:
{'meteor': 0.5277817049454638}

jaccard (%):
89.12

cosine (%):
77.14

levenshtein (%):
90.23

